# Cross-Country Flights: Price Prediction Modeling ✈️
## Notebook 3: Feature Engineering
### Will Bartlett & Kevin Tran

This notebook imports data from Dropbox and performs feature engineering, preparing the data for modeling. The goal of these models is to predict the price of a flight for the next day.

### 1. Get Data From Dropbox
<b>NOTE:</b> Gets all .csv files from Dropbox, not from the sample data in the <i>data</i> folder of the repository

In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dropbox
import os
import io
from dropbox.exceptions import ApiError
from scipy import stats
import datetime as dt
import re

In [116]:
# Save dropbox credentials
DROPBOX_APP_KEY = "9gghuhtkyrtyn2u"
DROPBOX_APP_SECRET = "tqtvwbr15gbwseg"
DROPBOX_REFRESH_TOKEN = "LpMKV__by1wAAAAAAAAAAYn_oi2fIW0SBBqF0n6gap1WavDnJGFRMcVWQol7X5Vy"

# Create Dropbox client
dbx = dropbox.Dropbox(
    app_key=DROPBOX_APP_KEY,
    app_secret=DROPBOX_APP_SECRET,
    oauth2_refresh_token=DROPBOX_REFRESH_TOKEN
)

print("Reading files from Dropbox...")

# List all files
result = dbx.files_list_folder('/flight_data')

dfs = []
for entry in result.entries:
    if entry.name.endswith('.csv'):
        print(f"Reading: {entry.name}")
        
        # Download file content
        metadata, response = dbx.files_download(entry.path_lower)
        
        # Read directly into pandas
        df = pd.read_csv(io.BytesIO(response.content))
        dfs.append(df)

# Combine
combined_df = pd.concat(dfs, ignore_index=True)

# Process
combined_df['route'] = combined_df['origin'] + ' → ' + combined_df['destination']
combined_df['time_collected'] = pd.to_datetime(combined_df['time_collected'])
combined_df['departure_date'] = pd.to_datetime(combined_df['departure_date'])
df = combined_df.copy()

# Convert to datetime with error handling
try:
    if df['departure_date'].dtype != 'datetime64[ns]':
        df['departure_date'] = pd.to_datetime(df['departure_date'], errors='coerce')
except Exception as e:
    print(f"Error with departure_date: {e}")
    # Try alternative parsing
    df['departure_date'] = pd.to_datetime(df['departure_date'].astype(str), errors='coerce')

try:
    if df['departure_time'].dtype != 'datetime64[ns]':
        df['departure_time'] = pd.to_datetime(df['departure_time'], errors='coerce')
except Exception as e:
    print(f"Error with departure_time: {e}")
    df['departure_time'] = pd.to_datetime(df['departure_time'].astype(str), errors='coerce')

try:
    if df['time_collected'].dtype != 'datetime64[ns]':
        df['time_collected'] = pd.to_datetime(df['time_collected'], errors='coerce')
except Exception as e:
    print(f"Error with time_collected: {e}")
    df['time_collected'] = pd.to_datetime(df['time_collected'].astype(str), errors='coerce')

# Round departure time to nearest 15 minutes (flights might vary by a few minutes)
df['departure_time_rounded'] = df['departure_time'].dt.floor('15min')

# Create flight ID for each unique flight
df['flight_id'] = (
    df['origin'].astype(str) + '_' + 
    df['destination'].astype(str) + '_' + 
    df['departure_date'].dt.strftime('%Y%m%d') + '_' + 
    df['departure_time_rounded'].dt.strftime('%H%M') + '_' + 
    df['airline'].astype(str)
)
df.head()

Reading files from Dropbox...
Reading: flight_data_20251110_210135.csv
Reading: flight_data_20251111_031726.csv
Reading: flight_data_20251111_130430.csv
Reading: flight_data_20251112_021940.csv
Reading: flight_data_20251112_130852.csv
Reading: flight_data_20251113_022155.csv
Reading: flight_data_20251113_130822.csv
Reading: flight_data_20251114_022042.csv
Reading: flight_data_20251114_130600.csv
Reading: flight_data_20251115_021721.csv
Reading: flight_data_20251115_125944.csv
Reading: flight_data_20251116_022726.csv
Reading: flight_data_20251116_125744.csv
Reading: flight_data_20251117_022452.csv
Reading: flight_data_20251117_132325.csv
Reading: flight_data_20251118_025158.csv
Reading: flight_data_20251118_130553.csv
Reading: flight_data_20251119_021933.csv
Reading: flight_data_20251119_130946.csv
Reading: flight_data_20251120_021830.csv
Reading: flight_data_20251120_130617.csv
Reading: flight_data_20251121_022030.csv
Reading: flight_data_20251121_130024.csv
Reading: flight_data_202511

,time_collected,origin,destination,departure_date,days_until_departure,price,currency,airline,number_of_stops,departure_time,arrival_time,total_duration,aircraft_type,cabin_class,bookable_seats,route,departure_time_rounded,flight_id
0,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,144.27,EUR,F9,1,2025-11-11 06:25:00,2025-11-11T12:25:00,PT9H,"32Q,32Q",ECONOMY,4,JFK → LAX,2025-11-11 06:15:00,JFK_LAX_20251111_0615_F9
1,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,149.13,EUR,F9,1,2025-11-11 07:59:00,2025-11-11T19:53:00,PT14H54M,"32Q,32Q",ECONOMY,4,JFK → LAX,2025-11-11 07:45:00,JFK_LAX_20251111_0745_F9
2,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,149.13,EUR,F9,1,2025-11-11 07:59:00,2025-11-11T20:42:00,PT15H43M,"32Q,32N",ECONOMY,4,JFK → LAX,2025-11-11 07:45:00,JFK_LAX_20251111_0745_F9
3,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,210.02,EUR,F9,0,2025-11-11 11:29:00,2025-11-11T14:32:00,PT6H3M,32Q,ECONOMY,4,JFK → LAX,2025-11-11 11:15:00,JFK_LAX_20251111_1115_F9
4,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,256.07,EUR,AS,1,2025-11-11 07:00:00,2025-11-11T15:17:00,PT11H17M,"73H,73J",ECONOMY,7,JFK → LAX,2025-11-11 07:00:00,JFK_LAX_20251111_0700_AS


### 2. Sort & Aggregate Data for Modeling

In [117]:
# Get date found from time_collected
df['search_date'] = pd.to_datetime(df['time_collected']).dt.date

# Remove non-economy flights
df = df[df['cabin_class'] == 'ECONOMY']

# Aggregate to daily averages for price
df_daily = df.groupby(['flight_id', 'search_date']).agg({
    'price': 'mean',
    'days_until_departure': 'first',  # Same for all obs that day
    'origin': 'first',
    'destination': 'first',
    'departure_date': 'first',
    'currency': 'first',
    'airline': 'first',
    'number_of_stops': 'first',
    'departure_time': 'first',
    'arrival_time': 'first',
    'total_duration': 'first',
    'aircraft_type': 'first',
    'cabin_class': 'first',
    'bookable_seats': 'first',
    'route': 'first',
    'departure_time_rounded': 'first'
}).reset_index()

# Convert search_date back to datetime for calculations
df_daily['search_date'] = pd.to_datetime(df_daily['search_date'])

# Convert price to USD (1 USD = 1.15 EUR)
df_daily['price'] = df_daily['price'] / 1.15
df_daily['currency'] = 'USD'

### 3. Create Lagged and Historical Price Features

In [118]:
# Lagged variables
df_daily = df_daily.sort_values(['flight_id', 'search_date'])
df_daily['price_lag1'] = df_daily.groupby('flight_id')['price'].shift(1)
df_daily['price_lag2'] = df_daily.groupby('flight_id')['price'].shift(2)
df_daily['price_lag3'] = df_daily.groupby('flight_id')['price'].shift(3)

# Price changes
df_daily['price_change_1day'] = df_daily['price'] - df_daily['price_lag1']
df_daily['price_pct_change_1day'] = (
    (df_daily['price'] - df_daily['price_lag1']) / df_daily['price_lag1']
)
df_daily['price_change_3day'] = df_daily['price'] - df_daily['price_lag3']

# Days since last observation (handles gaps!)
df_daily['days_since_last_obs'] = (
    df_daily.groupby('flight_id')['search_date'].diff().dt.days
)
df_daily['days_since_last_obs'] = df_daily['days_since_last_obs'].fillna(0)

# Rolling window statistics (3-observation window)
df_daily['price_rolling_mean_3'] = (
    df_daily.groupby('flight_id')['price']
    .rolling(window=3, min_periods=1).mean()
    .reset_index(level=0, drop=True)
)
df_daily['price_rolling_std_3'] = (
    df_daily.groupby('flight_id')['price']
    .rolling(window=3, min_periods=1).std()
    .reset_index(level=0, drop=True)
)

# First observed price (baseline)
df_daily['price_first_seen'] = (
    df_daily.groupby('flight_id')['price'].transform('first')
)
df_daily['price_change_since_first'] = df_daily['price'] - df_daily['price_first_seen']

# Create target (tomorrow's price)
df_daily['price_tomorrow'] = df_daily.groupby('flight_id')['price'].shift(-1)
df_daily['price_change_tomorrow'] = df_daily['price_tomorrow'] - df_daily['price']
df_daily['price_pct_change_tomorrow'] = (
    (df_daily['price_tomorrow'] - df_daily['price']) / df_daily['price']
)

# Add temporal features from departure_date
df_daily['departure_date'] = pd.to_datetime(df_daily['departure_date'])
df_daily['departure_day_of_week'] = df_daily['departure_date'].dt.dayofweek
df_daily['departure_month'] = df_daily['departure_date'].dt.month
df_daily['is_departure_weekend'] = df_daily['departure_day_of_week'].isin([4, 5, 6]).astype(int)

# Search date features
df_daily['search_day_of_week'] = df_daily['search_date'].dt.dayofweek
df_daily['search_month'] = df_daily['search_date'].dt.month

# Drop rows where target is missing (last observation per flight)
df_daily = df_daily.dropna(subset=['price_tomorrow']).copy()

### 4. Route-Based Features

In [119]:
# Number of airlines and flights per route
route_competition = df_daily.groupby('route').agg({
    'airline': 'nunique',
    'flight_id': 'nunique'
}).rename(columns={'airline': 'num_airlines_on_route', 'flight_id': 'unique_flights_per_route'})

df_daily = df_daily.merge(route_competition, on='route', how='left')

# Competition indicators
df_daily['is_high_competition'] = (df_daily['num_airlines_on_route'] >= 4).astype(int)

# East coast and west coast airports
east_coast = ['JFK', 'BOS', 'PHL']
west_coast = ['LAX', 'SFO', 'SEA']

# Create route direction
def get_route_direction(row):
    if row['origin'] in east_coast and row['destination'] in west_coast:
        return 'east_to_west'
    elif row['origin'] in west_coast and row['destination'] in east_coast:
        return 'west_to_east'
    else:
        return 'other'

df_daily['route_direction'] = df_daily.apply(get_route_direction, axis=1)
df_daily['is_east_to_west'] = (df_daily['route_direction'] == 'east_to_west').astype(int)


# Flight Time Features
def parse_duration(duration_str):
    if pd.isna(duration_str):
        return None
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?', str(duration_str))
    if match:
        hours = int(match.group(1) or 0)
        minutes = int(match.group(2) or 0)
        return hours * 60 + minutes
    return None

df_daily['duration_minutes'] = df_daily['total_duration'].apply(parse_duration)

# Long haul indicator (>6 hours)
df_daily['is_long_haul'] = (df_daily['duration_minutes'] > 360).astype(int)

# Calculate route minimum duration
route_min_duration = df_daily.groupby('route')['duration_minutes'].min()
df_daily['route_min_duration'] = df_daily['route'].map(route_min_duration)

# Duration vs route minimum (indicates connections/layovers)
df_daily['duration_vs_route_min'] = df_daily['duration_minutes'] - df_daily['route_min_duration']

# Route-level volatility 
route_volatility = df_daily.groupby('route')['price'].std()
df_daily['route_price_volatility'] = df_daily['route'].map(route_volatility)
df_daily['is_high_volatility_route'] = (df_daily['route_price_volatility'] > 150).astype(int)

### 5. Airline-Based Features

In [120]:
# Airline pricing tier 
airline_tiers = {
    'HA': 'premium',      # Hawaiian Airlines
    'AS': 'premium',      # Alaska Airlines
    'UA': 'mid_cost',     # United Airlines 
    'B6': 'low_cost',     # JetBlue 
    'F9': 'ultra_low_cost'  # Frontier 
}
df_daily['airline_tier'] = df_daily['airline'].map(airline_tiers)

# Encode for interactions
tier_encoding = {'ultra_low_cost': 1, 'low_cost': 2, 'mid_cost': 3, 'premium': 4}
df_daily['airline_tier_encoded'] = df_daily['airline_tier'].map(tier_encoding)

# Volatility
airline_volatility = df_daily.groupby('airline')['price'].std()
df_daily['airline_price_volatility'] = df_daily['airline'].map(airline_volatility)
df_daily['is_high_volatility_airline'] = (df_daily['airline_price_volatility'] > 150).astype(int)

### 6. Bookable Seats Features

In [121]:
# Scarcity indicator (1-3 seats)
df_daily['is_scarce'] = (df_daily['bookable_seats'] <= 3).astype(int)

# Distance from the "4 seat anomaly" (you found weird drop at 4 seats)
df_daily['seats_distance_from_4'] = abs(df_daily['bookable_seats'] - 4)

# Is this the anomaly point?
df_daily['is_4_seat_anomaly'] = (df_daily['bookable_seats'] == 4).astype(int)

# Plenty of seats available (5+)
df_daily['seats_plenty'] = (df_daily['bookable_seats'] >= 5).astype(int)

### 7. Booking Window Features

In [122]:
# Panic booking zone (1-2 days, highest prices)
df_daily['is_panic_booking'] = (df_daily['days_until_departure'] <= 2).astype(int)

# Sweet spot zone (drop at 7 days out)
df_daily['is_sweet_spot'] = (
    (df_daily['days_until_departure'] >= 6) & 
    (df_daily['days_until_departure'] <= 8)
).astype(int)

# Early booking zone (10-14 days, prices rise again)
df_daily['is_early_booking'] = (df_daily['days_until_departure'] >= 10).astype(int)

# Distance from optimal (7 days)
df_daily['days_from_optimal'] = abs(df_daily['days_until_departure'] - 7)

### 8. Temporal Features

In [123]:
# Extract hour from departure_time
df_daily['departure_hour'] = pd.to_datetime(df_daily['departure_time']).dt.hour

# Red-eye indicator 
df_daily['is_redeye'] = (
    (df_daily['departure_hour'] >= 22) | (df_daily['departure_hour'] <= 5)
).astype(int)

# Departure time categories
def categorize_departure_time(hour):
    if hour >= 22 or hour <= 5:
        return 'redeye'
    elif hour <= 9:
        return 'early_morning'
    elif hour <= 17:
        return 'daytime'
    elif hour <= 21:
        return 'evening'
    else:
        return 'night'

df_daily['departure_time_category'] = df_daily['departure_hour'].apply(categorize_departure_time)

# Preferred departure times (6-9am, 5-9pm typically more expensive)
df_daily['is_preferred_departure_time'] = (
    ((df_daily['departure_hour'] >= 6) & (df_daily['departure_hour'] <= 9)) |
    ((df_daily['departure_hour'] >= 17) & (df_daily['departure_hour'] <= 21))
).astype(int)

# Cheap departure days (Mon-Wed)
df_daily['is_cheap_departure_day'] = df_daily['departure_day_of_week'].isin([0, 1, 2]).astype(int)

# Expensive departure days (Fri-Sun)
df_daily['is_expensive_departure_day'] = df_daily['departure_day_of_week'].isin([4, 5, 6]).astype(int)

### 9. Aircraft-Based Features

In [124]:
# Boeing vs Airbus
df_daily['is_boeing'] = df_daily['aircraft_type'].str.contains('Boeing|737|777|787', case=False, na=False).astype(int)
df_daily['is_airbus'] = df_daily['aircraft_type'].str.contains('Airbus|A320|A321', case=False, na=False).astype(int)

# Wide-body indicator (international/premium aircraft)
df_daily['is_widebody'] = df_daily['aircraft_type'].str.contains('777|787|A330|A350', case=False, na=False).astype(int)

### 10. Create Final Dataset

In [125]:
# Drop rows with nulls
df_daily = df_daily.dropna()

In [126]:
# Drop any unneccessary features before modeling - using PRICE TOMORROW as the target
drop_features = [
    'price_change_tomorrow',      # Derived from target
    'price_pct_change_tomorrow',  # Derived from target
    # ID and date columns
    'flight_id',
    'departure_date',
    'departure_time',
    'arrival_time',
    # Raw categorical variables
    'origin',
    'destination', 
    'airline',
    'currency',
    'airline_tier',  # Keep airline_tier_encoded
    'departure_time_category',  # Keep hour + binaries instead
    'price_first_seen',  # Less useful than current price
    'price_change_since_first', # Less useful than lags
    # Redundant binary indicators
    'is_expensive_departure_day',  # Keep is_cheap_departure_day
    'seats_plenty',               # Keep is_scarce
    'is_high_volatility_airline', # Keep continuous airline_price_volatility
    'is_high_volatility_route',   # Keep continuous route_price_volatility
    'search_day_of_week', 
    'departure_time_rounded',
    'route',
    'cabin_class',
    'aircraft_type',
    'days_since_last_obs',
    'total_duration',
    'price_change_1day',
    'price_change_3day',
    'bookable_seats',
    'number_of_stops',
    'search_month',
    'route_direction'
]
drop_features = [col for col in drop_features if col in df_daily.columns]
df_engineered = df_daily.drop(columns=drop_features).copy()

In [127]:
df_engineered.head()

,search_date,price,days_until_departure,price_lag1,price_lag2,price_lag3,price_pct_change_1day,price_rolling_mean_3,price_rolling_std_3,price_tomorrow,...,is_sweet_spot,is_early_booking,days_from_optimal,departure_hour,is_redeye,is_preferred_departure_time,is_cheap_departure_day,is_boeing,is_airbus,is_widebody
179,2025-11-13,329.147826,2,288.982609,288.982609,289.973913,0.138988,302.371014,23.189399,235.652174,...,0,0,5,5,1,0,0,0,0,0
183,2025-11-13,329.147826,2,288.982609,288.982609,289.973913,0.138988,302.371014,23.189399,235.652174,...,0,0,5,6,0,1,0,0,0,0
187,2025-11-13,341.924638,2,297.860870,297.860870,298.856522,0.147934,312.548792,25.440228,347.826087,...,0,0,5,7,0,1,0,0,0,0
191,2025-11-13,329.147826,2,288.982609,288.982609,289.973913,0.138988,302.371014,23.189399,235.652174,...,0,0,5,7,0,1,0,0,0,0
195,2025-11-13,341.924638,2,293.968116,293.968116,294.959420,0.163135,309.953623,27.687711,279.275362,...,0,0,5,7,0,1,0,0,0,0


In [128]:
df_engineered.shape

(123747, 38)

In [129]:
df_engineered.columns

Index(['search_date', 'price', 'days_until_departure', 'price_lag1',
       'price_lag2', 'price_lag3', 'price_pct_change_1day',
       'price_rolling_mean_3', 'price_rolling_std_3', 'price_tomorrow',
       'departure_day_of_week', 'departure_month', 'is_departure_weekend',
       'num_airlines_on_route', 'unique_flights_per_route',
       'is_high_competition', 'is_east_to_west', 'duration_minutes',
       'is_long_haul', 'route_min_duration', 'duration_vs_route_min',
       'route_price_volatility', 'airline_tier_encoded',
       'airline_price_volatility', 'is_scarce', 'seats_distance_from_4',
       'is_4_seat_anomaly', 'is_panic_booking', 'is_sweet_spot',
       'is_early_booking', 'days_from_optimal', 'departure_hour', 'is_redeye',
       'is_preferred_departure_time', 'is_cheap_departure_day', 'is_boeing',
       'is_airbus', 'is_widebody'],
      dtype='object')

In [130]:
# Save the engineered data; ignored in repository
df_engineered.to_csv('../data/features_engineered.csv', index=False)

### 